# Wikifonia XML-only preprocessing and MIDITok TSD encoding

This notebook builds the symbolic melody-to-chord dataset used in the dissertation. Melody notes and annotated chord symbols are extracted from the same compressed MusicXML lead sheet, aligned at 24 PPQ, and encoded with MIDITok TSD. The chord vocabulary is selected from the training split only; labels outside the retained Top-N set are mapped to `<UNK>`.


## Completion criteria and execution order

Run the notebook from top to bottom. The final audit verifies preprocessing counts, 12/24-PPQ quantisation safety, training-only Top-N selection, unknown-target rates, one supervised clock slot per grid step, absence of gold chord tokens from model input, song-group separation, and readiness for LSTM window construction.


> Notebook outputs have been cleared. Run the cells in order to reproduce the analysis; setup and data paths are listed in `docs/REPRODUCIBILITY.md`.


## 1. Install fixed package versions


In [ ]:
%pip -q install "music21==10.5.0" "harte-library==0.4.5" "miditok==3.0.6.post1" "symusic==0.6.0" "lark>=1.1,<2" "tqdm>=4.66" "pandas>=2.0" "numpy>=1.26"


## 2. Imports, reproducibility settings and Top-N policy


In [ ]:
import gzip
import hashlib
import importlib.metadata
import json
import math
import os
import re
import subprocess
import sys
import warnings
import zipfile
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import music21
from IPython.display import display
from miditok import TSD, TokenizerConfig
from music21 import converter, harmony, note, pitch, repeat, stream
from harte.harte import Harte
from symusic import Note as SymusicNote
from symusic import Score as SymusicScore
from symusic import Track as SymusicTrack
from tqdm.auto import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


PIPELINE_REVISION = "XML_ONLY_V1_2026-08-04"


ENCODING_REVISION = "XML_ONLY_MIDITOK_TSD24_TOPN_UNK_V3_2026-08-19"
SELECTED_PPQ = 24
PPQ_CANDIDATES = (12, 24)
PREPROCESS_BATCH_SIZE = 64
STRIP_CHORD_INVERSIONS = True


TOP_N_RETAINED_CHORDS = 200
COVERAGE_TOP_N_VALUES = (50, 100, 150, 200, 250, 300)
VOCAB_REDUCTION_STRATEGY = "top_n_to_unk"



VOCAB_RANKING_BASIS = "supervised_slot_frequency"


ALWAYS_RETAIN_CHORDS = ("N",)

if TOP_N_RETAINED_CHORDS <= 0:
    raise ValueError('TOP_N_RETAINED_CHORDS must be greater than zero.')
if not 200 <= TOP_N_RETAINED_CHORDS <= 300:
    warnings.warn(
        'The current Top-N value is outside the suggested range of 200 to 300; '
        'ignore this warning if the value is intentional for an ablation.'
    )
if VOCAB_REDUCTION_STRATEGY != "top_n_to_unk":
    raise ValueError('This notebook currently implements only top_n_to_unk.')
if VOCAB_RANKING_BASIS not in {
    "supervised_slot_frequency",
    "chord_event_frequency",
}:
    raise ValueError('Unsupported VOCAB_RANKING_BASIS.')

print("Preprocessing revision:", PIPELINE_REVISION)
print("Encoding revision:", ENCODING_REVISION)
print("Top-N retained chords:", TOP_N_RETAINED_CHORDS)
print("Reduction strategy:", VOCAB_REDUCTION_STRATEGY)
print("Ranking basis:", VOCAB_RANKING_BASIS)
print("Python:", sys.version.split()[0])
print("music21:", music21.__version__)
print("harte-library:", importlib.metadata.version("harte-library"))
print("MIDITok:", importlib.metadata.version("miditok"))
print("symusic:", importlib.metadata.version("symusic"))

## 3. Mount storage and configure output paths


In [ ]:
IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Set this environment variable before running the notebook to reuse old artifacts.
output_override = os.environ.get("XML_ONLY_OUTPUT_ROOT")
if output_override:
    OUTPUT_ROOT = Path(output_override).expanduser().resolve()
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/melody-to-chord/artifacts")
else:
    OUTPUT_ROOT = PROJECT_ROOT / "artifacts"

CHOCO_DATA_DIR = Path(os.environ.get(
    "CHOCO_DATA_DIR", str(PROJECT_ROOT / "data" / "choco")
)).expanduser().resolve()
RAW_DIR_OVERRIDE = os.environ.get("WIKIFONIA_RAW_DIR") or None

PREPROCESS_ROOT = OUTPUT_ROOT / "preprocessing"
PREPROCESS_SHARD_DIR = PREPROCESS_ROOT / "shards"

# Separate directories prevent a Top-250 run overwriting Top-200 artifacts.
ENCODING_ROOT = OUTPUT_ROOT / f"encoding_miditok_tsd24_top{TOP_N_RETAINED_CHORDS}_unk_v3"
ENCODING_SHARD_DIR = ENCODING_ROOT / "shards"
for directory in [OUTPUT_ROOT, PREPROCESS_ROOT, PREPROCESS_SHARD_DIR,
                  ENCODING_ROOT, ENCODING_SHARD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)
print("Top-N encoding root:", ENCODING_ROOT)


In [ ]:
# The raw scores stay outside Git. A custom raw directory skips this download.
if RAW_DIR_OVERRIDE is None:
    if not (CHOCO_DATA_DIR / ".git").exists():
        CHOCO_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "https://github.com/smashub/choco.git", str(CHOCO_DATA_DIR)],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(CHOCO_DATA_DIR), "sparse-checkout", "set",
         "partitions/wikifonia", "choco/converters"],
        check=True,
    )
    RAW_DIR_OVERRIDE = str(CHOCO_DATA_DIR / "partitions" / "wikifonia" / "raw")

raw_path = Path(RAW_DIR_OVERRIDE).expanduser().resolve()
mxl_count = sum(
    1 for path in raw_path.rglob("*")
    if path.is_file() and re.search(r"\.mxl(?:\.\d+)?$", path.name, re.IGNORECASE)
)
print("MXL source:", raw_path)
print("MXL files found:", mxl_count)
if mxl_count == 0:
    raise RuntimeError("No MXL files found; check WIKIFONIA_RAW_DIR or the ChoCo download.")


## 4. Locate Wikifonia MusicXML files and the ChoCo converter


In [ ]:
raw_candidates = [CHOCO_DATA_DIR / "partitions" / "wikifonia" / "raw"]

if RAW_DIR_OVERRIDE is not None:
    raw_candidates.insert(0, Path(RAW_DIR_OVERRIDE))

RAW_DIR = next(
    (path for path in raw_candidates if path.exists()),
    None,
)
if RAW_DIR is None:
    raise FileNotFoundError(
        "The Wikifonia MXL directory was not found. Set WIKIFONIA_RAW_DIR "
        "before running the notebook and rerun the path configuration."
    )

def infer_repo_root(raw_dir):
    raw_dir = Path(raw_dir).resolve()
    for parent in [raw_dir, *raw_dir.parents]:
        if (parent / ".git").exists():
            # A custom raw directory can be inside this portfolio repository.
            # Never apply sparse-checkout operations to an unrelated parent repo.
            remote = subprocess.run(
                ["git", "-C", str(parent), "remote", "get-url", "origin"],
                capture_output=True, text=True, check=False,
            )
            origin = remote.stdout.strip().removesuffix(".git").rstrip("/")
            if origin.endswith(("github.com/smashub/choco", "github.com:smashub/choco")):
                return parent
    return None

DATA_REPO_ROOT = infer_repo_root(RAW_DIR)
converter_file = None

if DATA_REPO_ROOT is not None:
    candidate = (
        DATA_REPO_ROOT
        / "choco"
        / "converters"
        / "chord_converter.py"
    )
    if not candidate.exists():
        subprocess.run(
            [
                "git", "-C", str(DATA_REPO_ROOT),
                "sparse-checkout", "add", "choco/converters",
            ],
            check=True,
        )
    if candidate.exists():
        converter_file = candidate

if converter_file is None:
    CHOCO_CODE_ROOT = PROJECT_ROOT / "data" / "choco-converter"
    candidate = (
        CHOCO_CODE_ROOT
        / "choco"
        / "converters"
        / "chord_converter.py"
    )
    if not candidate.exists():
        if CHOCO_CODE_ROOT.exists():
            raise RuntimeError(
                f"{CHOCO_CODE_ROOT} exists, but the converter is incomplete; "
                'Delete the temporary directory and rerun this cell.'
            )
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--filter=blob:none", "--sparse",
                "https://github.com/smashub/choco.git",
                str(CHOCO_CODE_ROOT),
            ],
            check=True,
        )
        subprocess.run(
            [
                "git", "-C", str(CHOCO_CODE_ROOT),
                "sparse-checkout", "set", "choco/converters",
            ],
            check=True,
        )
    converter_file = candidate

if not converter_file.exists():
    raise FileNotFoundError('ChoCo chord_converter.py is still unavailable.')

# ChoCo compatibility patch for Python 3.13
legacy_import = "from lib2to3.pytree import convert"
converter_source = converter_file.read_text(encoding="utf-8")

if legacy_import in converter_source:
    converter_source = converter_source.replace(
        legacy_import,
        "# Removed unused lib2to3 import for Python 3.13 compatibility",
        1,
    )
    converter_file.write_text(
        converter_source,
        encoding="utf-8",
    )
    print("ChoCo Python 3.13 compatibility patch applied")
else:
    print("ChoCo compatibility patch already applied or not required")


sys.modules.pop("chord_converter", None)
sys.modules.pop("harte_utils", None)

CHOCO_CONVERTER_DIR = converter_file.parent
if str(CHOCO_CONVERTER_DIR) not in sys.path:
    sys.path.insert(0, str(CHOCO_CONVERTER_DIR))

from chord_converter import ChordConverter
from harte_utils import calculate_interval, convert_root, simplify_harte

CHORD_CONVERTER = ChordConverter(
    "wikifonia",
    handle_error=False,
)

print("MXL source:", RAW_DIR)
print("ChoCo converter:", converter_file)
print("JAMS input used: False")

# This records the current checkout; the submission archive did not retain its commit.
repo_for_provenance = infer_repo_root(converter_file.parent)
commit_result = subprocess.run(
    ["git", "-C", str(repo_for_provenance), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
)
provenance = {
    "upstream_repository": "https://github.com/smashub/choco",
    "converter_git_commit": commit_result.stdout.strip(),
    "raw_data_git_commit": (
        commit_result.stdout.strip() if DATA_REPO_ROOT is not None else None
    ),
    "raw_data_directory": str(RAW_DIR.resolve()),
    "python_version": sys.version,
    "packages": {
        package: importlib.metadata.version(package)
        for package in ["music21", "harte-library", "miditok", "symusic", "numpy", "pandas"]
    },
    "converter_compatibility_patch": "remove unused lib2to3 import if present",
}
(OUTPUT_ROOT / "data_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)
print("Saved data_provenance.json")


## 5. Build the XML-only source manifest

Each source receives a stable record identifier. Filename variants such as `.mxl` and `.mxl.1` are assigned to the same song group to prevent leakage across splits.


In [ ]:
MXL_NAME_PATTERN = re.compile(r"\.mxl(?:\.\d+)?$", re.IGNORECASE)

mxl_files = sorted(
    path
    for path in RAW_DIR.rglob("*")
    if path.is_file() and MXL_NAME_PATTERN.search(path.name)
)
if not mxl_files:
    raise RuntimeError(f"{RAW_DIR} contains no .mxl files")

def stable_record_id(relative_path): 
    return hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:20]

def canonical_group_id(relative_path): 
    name = MXL_NAME_PATTERN.sub("", Path(relative_path).name)
    name = re.sub(r"\s+", " ", name).strip().casefold()
    return name

source_rows = []
for source_index, path in enumerate(mxl_files):
    relative_path = path.relative_to(RAW_DIR).as_posix()
    source_rows.append(
        {
            "source_index": source_index,
            "record_id": stable_record_id(relative_path),
            "relative_path": relative_path,
            "file_name": path.name,
            "group_id": canonical_group_id(relative_path),
            "absolute_path": str(path),
            "file_size_bytes": path.stat().st_size,
        }
    )

source_df = pd.DataFrame(source_rows)
assert source_df["record_id"].is_unique

display(source_df.head())
print("MXL files:", len(source_df))
print("Unique song groups:", source_df["group_id"].nunique())


## 6. Normalise chord labels

ChoCo's Wikifonia converter is used first. If a MusicXML figure cannot be parsed by its grammar, a Harte-compatible interval label is constructed from the parsed root, bass and pitch content and validated with `harte-library`.


In [ ]:
CANONICAL_ROOTS = [
    "C", "C#", "D", "Eb", "E", "F",
    "F#", "G", "Ab", "A", "Bb", "B",
]

@lru_cache(maxsize=None)
def primary_figure_to_harte(figure):
    converted = CHORD_CONVERTER.convert_chords(str(figure)) 
    return Harte(converted).prettify()

def pitch_fallback_to_harte(symbol): 
    root = symbol.root()
    bass = symbol.bass()
    if root is None or bass is None or not symbol.pitchNames:
        raise ValueError(
            f"Could not obtain root, bass or pitches from ChordSymbol: {symbol.figure}"
        )

    converted_root = convert_root(root)
    bass_interval = calculate_interval(root, bass, simple=True)
    bass_suffix = (
        f"/{bass_interval}" if bass_interval != "1" else ""
    )
    intervals = [
        calculate_interval(
            note.Note(root),
            note.Note(pitch_name),
            simple=True,
        )
        for pitch_name in symbol.pitchNames
    ]
    if "1" in intervals:
        intervals.remove("1")
    elif "8" in intervals:
        intervals.remove("8")

    explicit_degrees = simplify_harte(intervals)
    candidate = converted_root + explicit_degrees + bass_suffix
    return Harte(candidate).prettify() 

def canonicalize_harte_root(label):
    label = str(label).strip()
    if label == "N":
        return "N"
    root, remainder = (
        label.split(":", 1)
        if ":" in label
        else (label, "")
    )
    canonical_root = CANONICAL_ROOTS[
        pitch.Pitch(root).pitchClass
    ]
    normalized = (
        f"{canonical_root}:{remainder}"
        if remainder
        else canonical_root
    )
    return Harte(normalized).prettify() 

def chord_symbol_to_harte(symbol):
    if isinstance(symbol, harmony.NoChord):
        return "N", "no_chord"

    figure = str(symbol.figure).strip()
    if figure in {"", "N", "NC", "N.C."}:
        return "N", "no_chord"

    try: 
        label = primary_figure_to_harte(figure)
        method = "choco_figure_converter"
    except Exception:
        label = pitch_fallback_to_harte(symbol)
        method = "music21_pitch_fallback"

    if STRIP_CHORD_INVERSIONS and "/" in label:
        label = label.split("/", 1)[0]
    label = canonicalize_harte_root(label)
    return label, method

conversion_examples = [
    "Cmaj7", "Bm7b5", "F#7#9", "B-", "C/E", "Cadd9"
]
for figure in conversion_examples:
    symbol = harmony.ChordSymbol(figure)
    label, method = chord_symbol_to_harte(symbol)
    print(f"{figure:8s} → {label:12s} ({method})")


## 7. Select the lead-sheet part and expand repeats

Only a part containing both melody notes and chord symbols is accepted. Repeat expansion is applied to the complete part so that melody and harmony remain aligned.


In [ ]:
class NoJointMelodyChordPartError(ValueError):
    pass

class PolyphonicMelodyError(ValueError):
    pass

class AmbiguousXmlChordError(ValueError):
    pass

def choose_leadsheet_part(score): 
    candidates = []
    for part_index, part in enumerate(score.parts): 
        chord_count = len(  
            part.recurse().getElementsByClass(
                harmony.ChordSymbol
            )
        )
        note_count = sum(  
            isinstance(element, note.Note)
            for element in part.flatten().notes
        )
        if chord_count > 0 and note_count > 0:  
            candidates.append(
                (chord_count, note_count, -part_index, part)
            )

    if not candidates: 
        raise NoJointMelodyChordPartError(
            'No single part contains both melody notes and ChordSymbol objects.'
        )
    return max(candidates, key=lambda item: item[:3])[-1]

def expand_same_source_part(raw_part): ##repeat expansion EX:A → B → repeat to A → B → A → B
    try:
        expanded = repeat.Expander(raw_part).process()
        if float(expanded.highestTime) <= 0:
            raise ValueError('Repeat expansion produced an empty timeline.')
        return expanded, "expander_success"
    except repeat.ExpanderException:
        return raw_part, "original_repeat_inconsistent"


## 8. Extract monophonic melody events

Ties are consolidated before events are stored as onset, end and MIDI pitch. Genuinely overlapping events are rejected rather than silently reduced to an arbitrary voice.


In [ ]:
TIME_TOLERANCE = 1e-8

def extract_melody_events(part): 
    try:
        melody_part = part.stripTies( 
            inPlace=False,
            matchByPitch=True,
        )
    except Exception:
        melody_part = part

    raw_events = []
    skipped_nonpositive = 0
    for element in melody_part.flatten().notes: 
        if not isinstance(element, note.Note):
            continue
        onset = float(element.offset) 
        duration = float(element.quarterLength) 
        end = onset + duration
        if duration <= TIME_TOLERANCE:
            skipped_nonpositive += 1
            continue
        if onset < -TIME_TOLERANCE:
            raise ValueError(f"Melody onset is negative: {onset}")
        raw_events.append( 
            {
                "onset_ql": max(0.0, onset),
                "end_ql": end,
                "midi": int(element.pitch.midi),
            }
        )

    if not raw_events:
        raise ValueError('No valid melody notes remain after tie processing.')

    unique = {
        (
            round(event["onset_ql"], 10),
            round(event["end_ql"], 10),
            event["midi"],
        ): event
        for event in raw_events
    }
    events = sorted(
        unique.values(),
        key=lambda event: (
            event["onset_ql"],
            event["end_ql"],
            event["midi"],
        ),
    )

    previous_end = -math.inf 
    for event in events:
        if event["onset_ql"] < previous_end - TIME_TOLERANCE:
            raise PolyphonicMelodyError(
                'Melody events overlap on the quarterLength timeline.'
            )
        previous_end = max(previous_end, event["end_ql"])

    return events, skipped_nonpositive


## 9. Extract chord intervals

Chord symbols share the melody's absolute quarter-length time axis. Duplicate labels at the same onset are consolidated, conflicts are rejected, and an initial no-chord interval is inserted when the first annotation occurs after time zero.


In [ ]:
def extract_chord_intervals(part, score_length_ql): 
    grouped = {}
    method_counter = Counter()
    raw_symbol_count = 0

    for symbol in part.flatten().getElementsByClass(
        harmony.ChordSymbol
    ):
        raw_symbol_count += 1
        onset = float(symbol.offset)
        if onset < -TIME_TOLERANCE:
            raise ValueError(f"Chord onset is negative: {onset}")
        if onset >= score_length_ql - TIME_TOLERANCE:
            raise ValueError(
                'A chord onset occurs at or beyond the song end, so a positive duration cannot be constructed.'
            )

        label, method = chord_symbol_to_harte(symbol)
        method_counter[method] += 1
        onset_key = round(max(0.0, onset), 10)
        grouped.setdefault(onset_key, []).append( 
            {                                     
                "label": label,
                "source_figure": str(symbol.figure),
                "normalization_method": method,
            }
        )

    if not grouped:
        raise ValueError('The MXL file contains no ChordSymbol objects.')

    events = []
    for onset in sorted(grouped):
        items = grouped[onset]
        labels = sorted({item["label"] for item in items}) 
        if len(labels) != 1: 
            raise AmbiguousXmlChordError(
                f"Conflicting chords at the same XML onset={onset}: {labels}"
            )
        chosen = items[0]
        events.append(
            {
                "onset_ql": float(onset),
                "label": labels[0],
                "source_figure": chosen["source_figure"],
                "normalization_method": chosen[
                    "normalization_method"
                ],
            }
        )

    
    merged_events = []
    for event in events:
        if ( 
            merged_events
            and merged_events[-1]["label"] == event["label"]
        ):
            continue
        merged_events.append(event)

    if merged_events[0]["onset_ql"] > TIME_TOLERANCE: 
        merged_events.insert( 
            0,
            {
                "onset_ql": 0.0,
                "label": "N",
                "source_figure": "<synthetic_no_chord>",
                "normalization_method": "synthetic_no_chord",
            },
        )

    intervals = []
    for index, event in enumerate(merged_events):
        end = ( 
                
                
            merged_events[index + 1]["onset_ql"]
            if index + 1 < len(merged_events)
            else score_length_ql
        )
        if end <= event["onset_ql"] + TIME_TOLERANCE:
            raise ValueError('A chord interval has a non-positive duration.')
        intervals.append(
            {
                "onset_ql": event["onset_ql"],
                "end_ql": float(end),
                "label": event["label"],
                "source_figure": event["source_figure"],
                "normalization_method": event[
                    "normalization_method"
                ],
            }
        )

    return intervals, raw_symbol_count, dict(method_counter)


## 10. Audit 12- and 24-PPQ quantisation

A song is marked unsafe only when quantisation collapses a duration, merges distinct onsets, or creates a false overlap.


In [ ]:
def qstep(value_ql, ppq): 
                          

    return int(np.floor(float(value_ql) * ppq + 0.5))

def audit_quantization(melody_events, chord_intervals, ppq):
    all_boundaries = []
    for event in melody_events:
        all_boundaries.extend(
            [event["onset_ql"], event["end_ql"]]
        )
    for interval in chord_intervals:
        all_boundaries.extend(
            [interval["onset_ql"], interval["end_ql"]]
        )

    errors = [
        abs(value - qstep(value, ppq) / ppq)
        for value in all_boundaries
    ]

    quantized_melody = [
        (
            qstep(event["onset_ql"], ppq),
            qstep(event["end_ql"], ppq),
        )
        for event in melody_events
    ]
    quantized_chords = [
        (
            qstep(interval["onset_ql"], ppq),
            qstep(interval["end_ql"], ppq),
        )
        for interval in chord_intervals
    ]

    collapsed_note_count = sum( 
        end <= start for start, end in quantized_melody
    )
    collapsed_chord_count = sum(
        end <= start for start, end in quantized_chords
    )

    
    melody_onsets = [start for start, _ in quantized_melody]
    melody_onset_collision_count = (
        len(melody_onsets) - len(set(melody_onsets))
    )

    melody_state_collision_count = sum( 
        current_start < previous_end
        for (_, previous_end), (current_start, _) in zip(
            quantized_melody,
            quantized_melody[1:],
        )
    )
    chord_state_collision_count = sum(
        current_start < previous_end
        for (_, previous_end), (current_start, _) in zip(
            quantized_chords,
            quantized_chords[1:],
        )
    )

    unsafe = any(
        [
            collapsed_note_count,
            collapsed_chord_count,
            melody_onset_collision_count,
            melody_state_collision_count,
            chord_state_collision_count,
        ]
    )
    return {
        "ppq": int(ppq),
        "exact": bool(max(errors, default=0.0) < 1e-9),
        "max_boundary_error_ql": float(
            max(errors, default=0.0)
        ),
        "collapsed_note_count": int(collapsed_note_count),
        "collapsed_chord_count": int(collapsed_chord_count),
        "melody_onset_collision_count": int(
            melody_onset_collision_count
        ),
        "melody_state_collision_count": int(
            melody_state_collision_count
        ),
        "chord_state_collision_count": int(
            chord_state_collision_count
        ),
        "unsafe": bool(unsafe),
    }


## 11. Preprocess one MusicXML song


In [ ]:
def assert_not_git_lfs_pointer(path):
    path = Path(path)
    if path.stat().st_size < 512:
        prefix = path.read_bytes()[:200]
        if b"version https://git-lfs.github.com/spec" in prefix:
            raise ValueError('The MXL file is still a Git LFS pointer and its content has not been downloaded.')

def parse_mxl_score(mxl_path): 
    """Parse both ordinary .mxl and versioned names such as .mxl.1."""
    mxl_path = Path(mxl_path)
    if mxl_path.suffix.lower() == ".mxl":
        return converter.parse(str(mxl_path), format="musicxml")

    # music21 uses the final suffix to choose the compressed-MusicXML
    # reader.  A Wikifonia backup named *.mxl.1 therefore needs its
    # internal XML passed explicitly instead of being misread as text.
    if zipfile.is_zipfile(mxl_path):
        with zipfile.ZipFile(mxl_path) as archive:
            members = [
                name
                for name in archive.namelist()
                if name.lower().endswith((".xml", ".musicxml"))
                and "meta-inf" not in name.lower()
            ]
            if not members:
                raise ValueError(
                    'No MusicXML member was found in the versioned MXL archive.'
                )
            xml_text = archive.read(members[0]).decode(
                "utf-8", errors="replace"
            )
        return converter.parseData(xml_text, format="musicxml")

    xml_text = mxl_path.read_text(
        encoding="utf-8", errors="replace"
    )
    return converter.parseData(xml_text, format="musicxml")

def safe_metadata(score):
    metadata = getattr(score, "metadata", None)
    title = getattr(metadata, "title", None) if metadata else None
    composer = (
        getattr(metadata, "composer", None) if metadata else None
    )
    return title or "", composer or ""

def process_mxl_row(row):
    base = {
        "source_index": int(row["source_index"]),
        "record_id": str(row["record_id"]),
        "relative_path": str(row["relative_path"]),
        "file_name": str(row["file_name"]),
        "group_id": str(row["group_id"]),
    }

    try:
        mxl_path = Path(row["absolute_path"])
        assert_not_git_lfs_pointer(mxl_path)
        score = parse_mxl_score(mxl_path)
        raw_part = choose_leadsheet_part(score)
        part, repeat_status = expand_same_source_part(raw_part)

        melody_events, skipped_nonpositive = (
            extract_melody_events(part)
        )
        provisional_length = max(
            float(part.highestTime),
            max(event["end_ql"] for event in melody_events),
        )
        if provisional_length <= TIME_TOLERANCE:
            raise ValueError('The song has a non-positive quarterLength duration.')

        chord_intervals, raw_chord_count, method_counts = (
            extract_chord_intervals(part, provisional_length)
        )
        score_length_ql = max(
            provisional_length,
            chord_intervals[-1]["end_ql"],
        )

        quantization = {
            str(ppq): audit_quantization(
                melody_events,
                chord_intervals,
                ppq,
            )
            for ppq in PPQ_CANDIDATES
        }
        title, composer = safe_metadata(score)

        record = {
            **base,
            "title": title,
            "composer": composer,
            "repeat_status": repeat_status,
            "score_length_ql": float(score_length_ql),
            "melody_events": melody_events,
            "chord_intervals": chord_intervals,
            "normalization_method_counts": method_counts,
            "quantization": quantization,
        }

        audit = {
            **base,
            "preprocessing_status": "success",
            "error_type": "",
            "error_message": "",
            "repeat_status": repeat_status,
            "score_length_ql": float(score_length_ql),
            "melody_event_count": len(melody_events),
            "raw_chord_symbol_count": raw_chord_count,
            "chord_interval_count": len(chord_intervals),
            "skipped_nonpositive_note_count": skipped_nonpositive,
            "pitch_fallback_chord_count": method_counts.get(
                "music21_pitch_fallback", 0
            ),
        }
        for ppq in PPQ_CANDIDATES:
            q = quantization[str(ppq)]
            for key, value in q.items():
                if key != "ppq":
                    audit[f"q{ppq}_{key}"] = value
        return record, audit

    except Exception as error:
        audit = {
            **base,
            "preprocessing_status": "failure",
            "error_type": type(error).__name__,
            "error_message": str(error)[:1000],
            "repeat_status": "",
        }
        return None, audit


## 12. Pilot preflight

The pilot finds three successful songs and displays concrete failure causes before the full batch process begins.


In [ ]:
pilot_records = []
pilot_audits = []

for row in source_df.head(60).to_dict("records"):
    record, audit = process_mxl_row(row)
    pilot_audits.append(audit)
    if record is not None:
        pilot_records.append(record)
    if len(pilot_records) >= 3:
        break

pilot_audit_df = pd.DataFrame(pilot_audits)
display(
    pilot_audit_df[
        [
            "record_id", "file_name", "preprocessing_status",
            "error_type", "error_message",
        ]
    ]
)

if not pilot_records:
    raise RuntimeError(
        'No song succeeded among the first 60 candidates; fix the first concrete error shown above '
        'before starting the full run.'
    )

pilot_summary = pd.DataFrame(
    [
        {
            "file_name": record["file_name"],
            "repeat_status": record["repeat_status"],
            "melody_events": len(record["melody_events"]),
            "chord_intervals": len(record["chord_intervals"]),
            "12_ppq_unsafe": record["quantization"]["12"][
                "unsafe"
            ],
            "24_ppq_unsafe": record["quantization"]["24"][
                "unsafe"
            ],
        }
        for record in pilot_records
    ]
)
display(pilot_summary)
print("Pilot passed:", len(pilot_records), "songs")


## 13. Verify that notes and harmony originate from the same MusicXML source


In [ ]:
def read_musicxml_text(mxl_path):
    mxl_path = Path(mxl_path)
    try:
        with zipfile.ZipFile(mxl_path) as archive:
            members = [
                name
                for name in archive.namelist()
                if name.lower().endswith((".xml", ".musicxml"))
                and "meta-inf" not in name.lower()
            ]
            if not members:
                raise ValueError('No MusicXML member was found in the MXL archive.')
            return archive.read(members[0]).decode(
                "utf-8", errors="replace"
            )
    except zipfile.BadZipFile:
        return mxl_path.read_text(
            encoding="utf-8", errors="replace"
        )

sample_record = pilot_records[0]
sample_path = RAW_DIR / sample_record["relative_path"]
sample_xml_text = read_musicxml_text(sample_path)

print("Sample:", sample_record["file_name"])
print("Contains <note>:", "<note" in sample_xml_text)
print("Contains <harmony>:", "<harmony" in sample_xml_text)
print("Melody and chord source path identical:", sample_path)


## 14. Inspect 24-PPQ melody and chord alignment


In [ ]:
def preview_on_grid(record, ppq=SELECTED_PPQ, max_steps=120):
    length = qstep(record["score_length_ql"], ppq)
    melody = np.full(length, "REST", dtype=object)
    melody_onset = np.zeros(length, dtype=np.uint8)
    chords = np.full(length, "N", dtype=object)
    chord_onset = np.zeros(length, dtype=np.uint8)

    for event in record["melody_events"]:
        start = qstep(event["onset_ql"], ppq)
        end = qstep(event["end_ql"], ppq)
        melody[start:end] = pitch.Pitch(
            midi=int(event["midi"])
        ).nameWithOctave
        melody_onset[start] = 1

    for interval in record["chord_intervals"]:
        start = qstep(interval["onset_ql"], ppq)
        end = qstep(interval["end_ql"], ppq)
        chords[start:end] = interval["label"]
        chord_onset[start] = 1

    shown = min(length, max_steps)
    return pd.DataFrame(
        {
            "step": np.arange(shown),
            "quarter_time": np.arange(shown) / ppq,
            "melody": melody[:shown],
            "melody_onset": melody_onset[:shown],
            "chord": chords[:shown],
            "chord_onset": chord_onset[:shown],
        }
    )

print(sample_record["file_name"])
display(preview_on_grid(sample_record))


## 15. Checkpoint helpers for full preprocessing


In [ ]:
PREPROCESS_COLUMNS = [
    "source_index", "record_id", "relative_path", "file_name",
    "group_id", "preprocessing_status", "error_type",
    "error_message", "repeat_status", "score_length_ql",
    "melody_event_count", "raw_chord_symbol_count",
    "chord_interval_count", "skipped_nonpositive_note_count",
    "pitch_fallback_chord_count",
] + [
    f"q{ppq}_{name}"
    for ppq in PPQ_CANDIDATES
    for name in [
        "exact", "max_boundary_error_ql",
        "collapsed_note_count", "collapsed_chord_count",
        "melody_onset_collision_count",
        "melody_state_collision_count",
        "chord_state_collision_count", "unsafe",
    ]
]

def preprocess_paths(batch_index):
    stem = f"preprocess_{batch_index:04d}"
    return {
        "events": PREPROCESS_SHARD_DIR / f"{stem}_events.json.gz",
        "audit": PREPROCESS_SHARD_DIR / f"{stem}_audit.csv",
        "done": PREPROCESS_SHARD_DIR / f"{stem}_done.json",
    }

def atomic_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(temporary, path)

def atomic_json_gz(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with gzip.open(temporary, "wt", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False)
    os.replace(temporary, path)

def atomic_csv(dataframe, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    dataframe.to_csv(temporary, index=False)
    os.replace(temporary, path)

def load_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return json.load(handle)

def done_marker_is_valid(paths, expected_record_ids):
    if not all(paths[key].exists() for key in ["events", "audit", "done"]):
        return False
    try:
        marker = json.loads(
            paths["done"].read_text(encoding="utf-8")
        )
        return (
            marker["pipeline_revision"] == PIPELINE_REVISION
            and marker["record_ids"] == expected_record_ids
        )
    except Exception:
        return False

def run_preprocessing_batches(source_table):
    batch_count = math.ceil(
        len(source_table) / PREPROCESS_BATCH_SIZE
    )
    for batch_index in tqdm(
        range(batch_count),
        desc="XML-only preprocessing batches",
    ):
        start = batch_index * PREPROCESS_BATCH_SIZE
        stop = min(start + PREPROCESS_BATCH_SIZE, len(source_table))
        batch_df = source_table.iloc[start:stop]
        expected_ids = batch_df["record_id"].astype(str).tolist()
        paths = preprocess_paths(batch_index)

        if done_marker_is_valid(paths, expected_ids):
            continue

        records = []
        audits = []
        for row in batch_df.to_dict("records"):
            record, audit = process_mxl_row(row)
            audits.append(audit)
            if record is not None:
                records.append(record)

        audit_df = pd.DataFrame(audits).reindex(
            columns=PREPROCESS_COLUMNS
        )
        if audit_df["record_id"].astype(str).tolist() != expected_ids:
            raise RuntimeError(
                f"Batch {batch_index} audit IDs are out of order"
            )

        atomic_json_gz(records, paths["events"])
        atomic_csv(audit_df, paths["audit"])
        atomic_json(
            {
                "pipeline_revision": PIPELINE_REVISION,
                "batch_index": batch_index,
                "record_ids": expected_ids,
                "attempted_count": len(batch_df),
                "success_count": len(records),
            },
            paths["done"],
        )

    return batch_count


## 16. Run resumable XML-only preprocessing


In [ ]:
preprocess_batch_count = run_preprocessing_batches(source_df)
print("Expected preprocessing batches:", preprocess_batch_count)
print("All batches reached done marker.")


## 17. Consolidate quality-control results and select 24-PPQ-safe songs


In [ ]:
audit_frames = []
missing_batches = []
for batch_index in range(preprocess_batch_count):
    paths = preprocess_paths(batch_index)
    if not all(
        paths[key].exists() for key in ["events", "audit", "done"]
    ):
        missing_batches.append(batch_index)
        continue
    audit_frames.append(pd.read_csv(paths["audit"]))

if missing_batches:
    raise RuntimeError(
        f"Preprocessing batches are incomplete: {missing_batches[:20]}"
    )

preprocessing_manifest_df = pd.concat(
    audit_frames,
    ignore_index=True,
).sort_values("source_index").reset_index(drop=True)

assert len(preprocessing_manifest_df) == len(source_df)
assert preprocessing_manifest_df["record_id"].is_unique

preprocessing_success_mask = preprocessing_manifest_df[
    "preprocessing_status"
].eq("success")
safe_24_mask = preprocessing_manifest_df[ 
    "q24_unsafe"
].eq(False)

preprocessing_manifest_df["final_status"] = np.where(
    preprocessing_success_mask & safe_24_mask,
    "accepted",
    "rejected",
)
preprocessing_manifest_df["final_reason"] = np.where(
    ~preprocessing_success_mask,
    preprocessing_manifest_df["error_type"].fillna("unknown_error"),
    np.where(
        ~safe_24_mask,
        "unsafe_at_24ppq",
        "xml_only_preprocessing_and_24ppq_pass",
    ),
)

accepted_df = preprocessing_manifest_df[
    preprocessing_manifest_df["final_status"].eq("accepted")
].copy()
rejected_df = preprocessing_manifest_df[
    preprocessing_manifest_df["final_status"].eq("rejected")
].copy()

PREPROCESS_MANIFEST_PATH = (
    PREPROCESS_ROOT / "xml_only_v1_manifest.csv"
)
ACCEPTED_PATH = PREPROCESS_ROOT / "xml_only_v1_accepted.csv"
REJECTED_PATH = PREPROCESS_ROOT / "xml_only_v1_rejected.csv"
PREPROCESS_SUMMARY_PATH = (
    PREPROCESS_ROOT / "xml_only_v1_summary.json"
)

atomic_csv(preprocessing_manifest_df, PREPROCESS_MANIFEST_PATH)
atomic_csv(accepted_df, ACCEPTED_PATH)
atomic_csv(rejected_df, REJECTED_PATH)

successful_df = preprocessing_manifest_df[
    preprocessing_success_mask
]
q12_unsafe_count = int(successful_df["q12_unsafe"].eq(True).sum())
q24_unsafe_count = int(successful_df["q24_unsafe"].eq(True).sum())

failure_counts = (
    preprocessing_manifest_df.loc[
        ~preprocessing_success_mask,
        "error_type",
    ]
    .fillna("unknown_error")
    .value_counts()
    .to_dict()
)

preprocessing_summary = {
    "pipeline_revision": PIPELINE_REVISION,
    "data_source": "single_mxl_only",
    "jams_used": False,
    "source_mxl_count": int(len(source_df)),
    "preprocessing_success_count": int(
        preprocessing_success_mask.sum()
    ),
    "preprocessing_failure_count": int(
        (~preprocessing_success_mask).sum()
    ),
    "selected_ppq": SELECTED_PPQ,
    "q12_unsafe_song_count": q12_unsafe_count,
    "q24_unsafe_song_count": q24_unsafe_count,
    "accepted_count": int(len(accepted_df)),
    "rejected_count": int(len(rejected_df)),
    "pending_count": 0,
    "ready_for_encoding": bool(len(accepted_df) > 0),
    "strip_chord_inversions": STRIP_CHORD_INVERSIONS,
    "failure_type_counts": failure_counts,
}
atomic_json(preprocessing_summary, PREPROCESS_SUMMARY_PATH)

print(json.dumps(preprocessing_summary, indent=2))
display(
    pd.DataFrame(
        {
            "PPQ": [12, 24],
            "unsafe_songs": [q12_unsafe_count, q24_unsafe_count],
            "max_boundary_error_ql": [
                successful_df["q12_max_boundary_error_ql"].max(),
                successful_df["q24_max_boundary_error_ql"].max(),
            ],
        }
    )
)


# MIDITok TSD-24 encoding and training-only Top-N targets

The observable melody is represented with MIDITok TSD tokens. A clock token is inserted at every 24-PPQ grid step, while gold chord labels are attached only as supervision and never enter `input_ids`. Chords outside the training-derived Top-N vocabulary are mapped to `<UNK>`.


## 18. Split by song group and measure training-only chord coverage

Groups are assigned deterministically to training, validation and test splits. Only training targets determine chord frequency ranks. Both slot coverage and chord-event coverage are reported for candidate vocabulary sizes.


In [ ]:
if not preprocessing_summary["ready_for_encoding"]:
    raise RuntimeError('The XML-only quality-control stage is not ready for encoding.')

def deterministic_split(group_id):
    bucket = int( 

        hashlib.sha256( 
                        
                        
                        
                        
            str(group_id).encode("utf-8")
        ).hexdigest()[:12],
        16,
    ) % 100

    if bucket < 80: 
        return "train"
    if bucket < 90:
        return "validation"
    return "test"

split_df = accepted_df[
    [
        "source_index", "record_id", "relative_path",
        "file_name", "group_id",
    ]
].copy()
split_df["record_id"] = split_df["record_id"].astype(str)
split_df["split"] = split_df["group_id"].map( 
                                              ##EX：Autumn_Leaves.mxl Autumn_Leaves.mxl.1
                                              
                                              
    deterministic_split
)

group_split_nunique = split_df.groupby("group_id")[ 
    "split"
].nunique()
assert int(group_split_nunique.max()) == 1

accepted_ids = set(split_df["record_id"])
split_by_id = split_df.set_index("record_id")["split"].to_dict()





split_slot_counts = { 
    split_name: Counter()
    for split_name in ("train", "validation", "test")
}
split_event_counts = { 
    split_name: Counter()
    for split_name in ("train", "validation", "test")
}

event_record_ids = set()
max_melody_duration_steps = 1

for batch_index in tqdm(
    range(preprocess_batch_count),
    desc="Scan accepted events for train-only chord coverage",
):
    records = load_json_gz(
        preprocess_paths(batch_index)["events"]
    )
    for record in records:
        record_id = str(record["record_id"])
        if record_id not in accepted_ids:
            continue

        event_record_ids.add(record_id)
        split_name = split_by_id[record_id]

        for interval in record["chord_intervals"]:
            label = str(interval["label"])
            start = qstep(interval["onset_ql"], SELECTED_PPQ) 
            end = qstep(interval["end_ql"], SELECTED_PPQ)  
            slot_count = end - start 
            if slot_count <= 0:
                raise RuntimeError(
                    f"Accepted chord has a non-positive quantised duration: {record_id}"
                )
            split_slot_counts[split_name][label] += slot_count 
                                                        
                                                        

            split_event_counts[split_name][label] += 1  ##event coverage
                                                        

        for event in record["melody_events"]:
            duration_steps = (
                qstep(event["end_ql"], SELECTED_PPQ)
                - qstep(event["onset_ql"], SELECTED_PPQ)
            )
            if duration_steps <= 0:
                raise RuntimeError(
                    f"Accepted melody has a non-positive quantised duration: {record_id}"
                )
            max_melody_duration_steps = max(
                max_melody_duration_steps,
                duration_steps,
            )

if event_record_ids != accepted_ids:
    missing = sorted(accepted_ids - event_record_ids)
    extra = sorted(event_record_ids - accepted_ids)
    raise RuntimeError(
        f"Accepted/event IDs are inconsistent; missing={missing[:5]}, "
        f"extra={extra[:5]}"
    )

training_slot_counts = split_slot_counts["train"]
training_event_counts = split_event_counts["train"]
training_slot_total = int(sum(training_slot_counts.values()))
training_event_total = int(sum(training_event_counts.values()))

if training_slot_total <= 0 or training_event_total <= 0:
    raise RuntimeError('The training split contains no chord targets for frequency analysis.')


ranking_universe = set(training_slot_counts) | set(ALWAYS_RETAIN_CHORDS)
if VOCAB_RANKING_BASIS == "supervised_slot_frequency":
    ranked_training_labels = sorted(
        ranking_universe,
        key=lambda label: (
            -training_slot_counts[label],
            -training_event_counts[label],
            label,
        ),
    )
else:
    ranked_training_labels = sorted(
        ranking_universe,
        key=lambda label: (
            -training_event_counts[label],
            -training_slot_counts[label],
            label,
        ),
    )

ranking_position = {
    label: rank
    for rank, label in enumerate(ranked_training_labels)
}

def select_top_n_labels(top_n):
    'Retain specified semantic labels and fill the remaining Top-N positions using training-only ranks.'
    top_n = int(top_n)
    mandatory = list(dict.fromkeys(ALWAYS_RETAIN_CHORDS))
    if top_n < len(mandatory):
        raise ValueError('Top-N is smaller than the number of ALWAYS_RETAIN_CHORDS entries.')

    non_mandatory = [
        label
        for label in ranked_training_labels
        if label not in mandatory
    ]
    selected = mandatory + non_mandatory[
        : max(0, top_n - len(mandatory))
    ]
    selected = list(dict.fromkeys(selected))
    selected.sort(
        key=lambda label: (ranking_position.get(label, math.inf), label)
    )
    return tuple(selected)

coverage_rows = []
coverage_values = sorted(
    set((*COVERAGE_TOP_N_VALUES, TOP_N_RETAINED_CHORDS))
)
for top_n in coverage_values:
    selected_labels = set(select_top_n_labels(top_n))
    retained_slots = int(
        sum(training_slot_counts[label] for label in selected_labels)
    )
    retained_events = int(
        sum(training_event_counts[label] for label in selected_labels)
    )
    retained_count = len(selected_labels)
    coverage_rows.append(
        {
            "requested_top_n": int(top_n),
            "retained_chord_label_count": retained_count,
            "effective_supervised_classes_with_unk": retained_count + 1,
            "technical_vocab_size_with_pad": retained_count + 2,
            "train_slot_coverage_pct": (
                100.0 * retained_slots / training_slot_total
            ),
            "train_unk_slot_pct": (
                100.0 * (training_slot_total - retained_slots)
                / training_slot_total
            ),
            "train_chord_event_coverage_pct": ( 
                100.0 * retained_events / training_event_total
            ),
            "train_unk_chord_event_pct": (
                100.0 * (training_event_total - retained_events)
                / training_event_total
            ),
        }
    )

coverage_selection_df = pd.DataFrame(coverage_rows)

cumulative_slots = 0
cumulative_events = 0
default_retained_set = set(
    select_top_n_labels(TOP_N_RETAINED_CHORDS)
)
frequency_rows = []
for rank, label in enumerate(ranked_training_labels, start=1):
    slot_count = int(training_slot_counts[label])
    event_count = int(training_event_counts[label])
    cumulative_slots += slot_count
    cumulative_events += event_count
    frequency_rows.append(
        {
            "training_rank": rank,
            "chord_label": label,
            "slot_count": slot_count,
            "slot_share_pct": 100.0 * slot_count / training_slot_total,
            "cumulative_slot_coverage_pct": (
                100.0 * cumulative_slots / training_slot_total
            ),
            "chord_event_count": event_count,
            "chord_event_share_pct": (
                100.0 * event_count / training_event_total
            ),
            "cumulative_chord_event_coverage_pct": (
                100.0 * cumulative_events / training_event_total
            ),
            "retained_by_current_top_n": label in default_retained_set,
        }
    )

training_frequency_df = pd.DataFrame(frequency_rows)

CHORD_COVERAGE_PATH = (
    ENCODING_ROOT / "train_topn_chord_coverage_v3.csv"
)
TRAIN_CHORD_FREQUENCY_PATH = (
    ENCODING_ROOT / "train_chord_frequency_v3.csv"
)
atomic_csv(coverage_selection_df, CHORD_COVERAGE_PATH)
atomic_csv(training_frequency_df, TRAIN_CHORD_FREQUENCY_PATH)

display(split_df["split"].value_counts().rename("songs"))
display(coverage_selection_df.round(4))
display(training_frequency_df.head(30).round(4))

print("Raw chord labels observed in training:", len(training_slot_counts))
print("Current Top-N choice:", TOP_N_RETAINED_CHORDS)
print("Ranking source: training split only")
print("Ranking basis:", VOCAB_RANKING_BASIS)
print("Coverage table saved:", CHORD_COVERAGE_PATH)
print(
    "To change vocabulary size, set TOP_N_RETAINED_CHORDS and rerun "
    "the notebook from the path configuration onwards."
)

## 19. Freeze the Top-N mapping and initialise MIDITok

The output vocabulary contains `<PAD>`, `<UNK>` and the retained Harte labels. `<PAD>` is used only for array padding; `<UNK>` remains a valid supervised class.


In [ ]:
retained_chord_labels = select_top_n_labels( 
    TOP_N_RETAINED_CHORDS
)
retained_chord_set = set(retained_chord_labels)

if "N" not in retained_chord_set: 
    raise RuntimeError('The no-chord label N must be retained.')

def map_chord_label(label): 
    'Retain original labels within Top-N and map all others to the predictable <UNK> class.'
    label = str(label)
    return label if label in retained_chord_set else "<UNK>"

chord_to_id = {"<PAD>": 0, "<UNK>": 1} 
for label in retained_chord_labels:
    chord_to_id[label] = len(chord_to_id)

id_to_chord = {
    str(token_id): label
    for label, token_id in chord_to_id.items()
}
CHORD_PAD_ID = int(chord_to_id["<PAD>"])
CHORD_UNK_ID = int(chord_to_id["<UNK>"])



coverage_by_split_rows = []
for split_name in ("train", "validation", "test"): 
    slot_counts = split_slot_counts[split_name]
    event_counts = split_event_counts[split_name]
    total_slots = int(sum(slot_counts.values()))
    total_events = int(sum(event_counts.values()))
    retained_slots = int(
        sum(
            count
            for label, count in slot_counts.items()
            if label in retained_chord_set
        )
    )
    retained_events = int( 
        sum(
            count
            for label, count in event_counts.items()
            if label in retained_chord_set
        )
    )
    unk_slots = total_slots - retained_slots
    unk_events = total_events - retained_events
    coverage_by_split_rows.append(
        {
            "split": split_name,
            "raw_chord_label_count": int(len(slot_counts)),
            "total_slots": total_slots,
            "retained_slots": retained_slots,
            "unk_slots": unk_slots,
            "retained_slot_coverage_pct": (
                100.0 * retained_slots / total_slots
                if total_slots else 0.0
            ),
            "unk_slot_pct": (
                100.0 * unk_slots / total_slots
                if total_slots else 0.0
            ),
            "total_chord_events": total_events,
            "retained_chord_events": retained_events,
            "unk_chord_events": unk_events,
            "retained_chord_event_coverage_pct": (
                100.0 * retained_events / total_events
                if total_events else 0.0
            ),
            "unk_chord_event_pct": (
                100.0 * unk_events / total_events
                if total_events else 0.0
            ),
        }
    )

coverage_by_split_df = pd.DataFrame(coverage_by_split_rows)

all_raw_labels = sorted( 
    set().union(
        *(set(counter) for counter in split_slot_counts.values())
    )
)
mapping_rows = []
for label in all_raw_labels:
    mapped_label = map_chord_label(label)
    mapping_rows.append(
        {
            "raw_harte_label": label,
            "training_rank": (
                ranking_position[label] + 1
                if label in ranking_position
                else ""
            ),
            "train_slot_count": int(training_slot_counts[label]),
            "train_chord_event_count": int(training_event_counts[label]),
            "retained": label in retained_chord_set,
            "mapped_target_label": mapped_label,
            "mapped_target_id": int(chord_to_id[mapped_label]),
            "validation_slot_count": int(
                split_slot_counts["validation"][label]
            ),
            "test_slot_count": int(split_slot_counts["test"][label]),
        }
    )
chord_mapping_df = pd.DataFrame(mapping_rows)

CHORD_MAPPING_PATH = (
    ENCODING_ROOT / "topn_unk_chord_mapping_v3.csv"
)
SPLIT_COVERAGE_PATH = (
    ENCODING_ROOT / "topn_unk_coverage_by_split_v3.csv"
)
atomic_csv(chord_mapping_df, CHORD_MAPPING_PATH)
atomic_csv(coverage_by_split_df, SPLIT_COVERAGE_PATH)


MIDITOK_MAX_DURATION_BEATS = max(
    16,
    int(math.ceil(
        max_melody_duration_steps / SELECTED_PPQ
    )),
)

miditok_config = TokenizerConfig(
    pitch_range=(0, 127),
    beat_res={ 
        (0, MIDITOK_MAX_DURATION_BEATS): SELECTED_PPQ,
    },
    num_velocities=1,
    special_tokens=["PAD", "BOS", "EOS", "MASK"],
    encode_ids_split="no",
    use_velocities=False,
    
    
    use_note_duration_programs=list(range(-1, 128)),
    use_chords=False, 
    use_rests=True,
    beat_res_rest={(0, 16): SELECTED_PPQ},
    use_tempos=False,
    use_time_signatures=False,
    use_programs=False,
    one_token_stream_for_programs=False,
    remove_duplicated_notes=False,
)
miditok_tokenizer = TSD(miditok_config) 

if miditok_tokenizer.time_division != SELECTED_PPQ: 
    raise RuntimeError(
        'MIDITok time division does not match the selected PPQ: '
        f"{miditok_tokenizer.time_division} != {SELECTED_PPQ}"
    )
if miditok_tokenizer.config.use_chords:
    raise RuntimeError('MIDITok chord detection must be disabled.')

MIDITOK_CLOCK_TOKEN = f"TimeShift_0.1.{SELECTED_PPQ}" 
required_miditok_tokens = [
    "PAD_None", "BOS_None", "EOS_None",
    MIDITOK_CLOCK_TOKEN,
]
missing_miditok_tokens = [
    token
    for token in required_miditok_tokens
    if token not in miditok_tokenizer.vocab
]
if missing_miditok_tokens:
    raise RuntimeError(
        f"MIDITok vocabulary is missing required tokens: {missing_miditok_tokens}"
    )

MIDITOK_PAD_ID = int(miditok_tokenizer["PAD_None"])
MIDITOK_BOS_ID = int(miditok_tokenizer["BOS_None"])
MIDITOK_EOS_ID = int(miditok_tokenizer["EOS_None"])
MIDITOK_CLOCK_ID = int(
    miditok_tokenizer[MIDITOK_CLOCK_TOKEN]
)

SPLIT_PATH = (
    ENCODING_ROOT / "xml_only_miditok_tsd24_topn_unk_v3_split.csv"
)
VOCAB_PATH = (
    ENCODING_ROOT / "xml_only_miditok_tsd24_topn_unk_v3_vocab.json"
)
MIDITOK_TOKENIZER_PATH = (
    ENCODING_ROOT / "miditok_tsd24_topn_unk_v3_tokenizer.json"
)

miditok_payload = miditok_tokenizer.to_dict()
miditok_tokenizer.save( 
    MIDITOK_TOKENIZER_PATH,
    additional_attributes={
        "encoding_revision": ENCODING_REVISION,
        "task": "melody_to_chord_sequence_labelling",
        "alignment": "one TimeShift_0.1.24 clock per 24-PPQ step",
        "gold_chords_in_input": False,
        "chord_reduction_strategy": VOCAB_REDUCTION_STRATEGY,
        "top_n_retained_chords": TOP_N_RETAINED_CHORDS,
        "vocabulary_selected_from_split": "train",
    },
)

coverage_by_split_records = json.loads(
    coverage_by_split_df.to_json(orient="records")
)
training_coverage = next(
    row
    for row in coverage_by_split_records
    if row["split"] == "train"
)

vocab_payload = { 
    "preprocessing_revision": PIPELINE_REVISION,
    "encoding_revision": ENCODING_REVISION,
    "ppq": SELECTED_PPQ,
    "representation": "MIDITok TSD-24 alignment-preserving expansion",
    "miditok": {
        "package_version": importlib.metadata.version("miditok"),
        "symusic_version": importlib.metadata.version("symusic"),
        "tokenizer_class": "TSD",
        "vocab_size": int(len(miditok_tokenizer)),
        "clock_token": MIDITOK_CLOCK_TOKEN,
        "clock_token_id": MIDITOK_CLOCK_ID,
        "pad_token_id": MIDITOK_PAD_ID,
        "bos_token_id": MIDITOK_BOS_ID,
        "eos_token_id": MIDITOK_EOS_ID,
        "max_duration_beats": MIDITOK_MAX_DURATION_BEATS,
        "use_chords": False, 
        "use_velocities": False,
    },
    "chord_target": {
        "source": "same MXL <harmony> converted to Harte",
        "reduction_strategy": VOCAB_REDUCTION_STRATEGY,
        "vocabulary_selected_from_split": "train",
        "ranking_basis": VOCAB_RANKING_BASIS,
        "requested_top_n": int(TOP_N_RETAINED_CHORDS),
        "raw_training_label_count": int(len(training_slot_counts)),
        "retained_label_count": int(len(retained_chord_labels)),
        "supervised_class_count_including_unk": int(
            len(retained_chord_labels) + 1
        ),
        "technical_vocab_size_including_pad": int(len(chord_to_id)),
        "training_retained_slot_coverage_pct": float(
            training_coverage["retained_slot_coverage_pct"]
        ),
        "training_retained_chord_event_coverage_pct": float(
            training_coverage["retained_chord_event_coverage_pct"]
        ),
        "onset_recomputed_after_mapping": True,
        "PAD": CHORD_PAD_ID,
        "UNK": CHORD_UNK_ID,
        
        "vocab_size": int(len(chord_to_id)),
        "retained_labels_in_rank_order": list(retained_chord_labels),
        "label_to_token": chord_to_id,
        "token_to_label": id_to_chord,
        "coverage_by_split": coverage_by_split_records,
        "mapping_path": str(CHORD_MAPPING_PATH),
    },
}
atomic_csv(split_df, SPLIT_PATH)
atomic_json(vocab_payload, VOCAB_PATH)

ENCODING_SCHEMA_SHA256 = hashlib.sha256( 
    json.dumps(
        {
            "encoding_revision": ENCODING_REVISION,
            "miditok": miditok_payload,
            "chord_to_id": chord_to_id,
            "reduction_strategy": VOCAB_REDUCTION_STRATEGY,
            "ranking_basis": VOCAB_RANKING_BASIS,
            "top_n": TOP_N_RETAINED_CHORDS,
            "retained_chord_labels": retained_chord_labels,
            "onset_recomputed_after_mapping": True,
        },
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()

display(coverage_by_split_df.round(4))
display(chord_mapping_df.head(30))
print("MIDITok used:", True)
print("MIDITok tokenizer:", type(miditok_tokenizer).__name__)
print("MIDITok vocabulary size:", len(miditok_tokenizer))
print("MIDITok clock:", MIDITOK_CLOCK_TOKEN, MIDITOK_CLOCK_ID)
print("MIDITok use_chords:", miditok_tokenizer.config.use_chords)
print("Max melody duration steps:", max_melody_duration_steps)
print("Configured max duration beats:", MIDITOK_MAX_DURATION_BEATS)
print("Raw training Harte labels:", len(training_slot_counts))
print("Retained Harte labels:", len(retained_chord_labels))
print(
    "Supervised chord classes (retained + <UNK>):",
    len(retained_chord_labels) + 1,
)
print(
    "Technical chord vocabulary (<PAD> + <UNK> + retained):",
    len(chord_to_id),
)
print(
    "Training slot coverage:",
    f"{training_coverage['retained_slot_coverage_pct']:.4f}%",
)
print(
    "Training chord-event coverage:",
    f"{training_coverage['retained_chord_event_coverage_pct']:.4f}%",
)
print("Vocabulary selected from validation/test: False")
print("Song-group leakage: 0")

## 20. Align standard MIDITok TSD events to the 24-PPQ clock

The standard compressed TSD sequence is retained for audit. The model stream expands time to one clock per grid step, attaches reduced chord targets only to clock positions, and recomputes chord onsets after rare-label mapping.


In [ ]:
def build_symusic_melody_score(record): 
    score = SymusicScore(SELECTED_PPQ) 
    track = SymusicTrack( 
        name="Wikifonia melody",
        program=0,
        is_drum=False,
    )
    for event in sorted( 
        record["melody_events"],
        key=lambda item: (
            float(item["onset_ql"]),
            int(item["midi"]),
        ),
    ):
        start = qstep(event["onset_ql"], SELECTED_PPQ) 
        end = qstep(event["end_ql"], SELECTED_PPQ)
        if not 0 <= start < end:
            raise ValueError(
                f"Invalid MIDITok melody range: {start}:{end}"
            )
        track.notes.append( 
            SymusicNote(
                start,
                end - start,
                int(event["midi"]),
                100,
            )
        )
    score.tracks.append(track)
    return score

def standard_miditok_tsd(record): 
    score = build_symusic_melody_score(record) 
    sequences = miditok_tokenizer.encode( 
        score,
        encode_ids=False,
    )
    if not isinstance(sequences, list) or len(sequences) != 1:
        raise ValueError(
            'The melody-only score was expected to produce one MIDITok sequence.'
        )
    sequence = sequences[0]

    tokens = list(sequence.tokens) 
    token_ids = np.asarray(sequence.ids, dtype=np.int32) ##Token IDs
    event_steps = np.asarray( 
        [int(round(float(event.time))) for event in sequence.events],
        dtype=np.int32,
    )
    
    token_types = [str(event.type_) for event in sequence.events]

    if not (
        len(tokens)
        == len(token_ids)
        == len(event_steps)
        == len(token_types)
    ):
        raise ValueError('MIDITok TokSequence fields have inconsistent lengths.')
    if len(token_ids) == 0:
        raise ValueError('MIDITok produced an empty sequence.')
    if np.any(token_ids == MIDITOK_PAD_ID):
        raise ValueError('A standard within-song MIDITok sequence must not contain PAD.')
    
    if any(token_type == "Chord" for token_type in token_types):
        raise ValueError('MIDITok input must not contain automatically detected chord tokens.')
    if np.any(np.diff(event_steps) < 0):
        raise ValueError('MIDITok event times are not monotonically non-decreasing.')

    expected_notes = [ 
        (
            qstep(event["onset_ql"], SELECTED_PPQ),
            qstep(event["end_ql"], SELECTED_PPQ)
            - qstep(event["onset_ql"], SELECTED_PPQ),
            int(event["midi"]),
        )
        for event in sorted(
            record["melody_events"],
            key=lambda item: (
                float(item["onset_ql"]),
                int(item["midi"]),
            ),
        )
    ]
    actual_notes = [
        (
            int(round(float(event.time))),
            int(round(float(event.desc)))
            - int(round(float(event.time))),
            int(event.value),
        )
        for event in sequence.events
        if event.type_ == "Pitch"
    ]
    if actual_notes != expected_notes:
        raise ValueError(
            'MIDITok did not preserve the 24-PPQ melody notes exactly; '
            f"expected={expected_notes[:3]}, actual={actual_notes[:3]}"
        )

    return {
        "tokens": tokens,
        "token_ids": token_ids,
        "event_steps": event_steps,
        "token_types": token_types,
    }

def chord_grid_for_record(record, sequence_length): 
    chord_grid = np.full(
        sequence_length,
        chord_to_id["N"],
        dtype=np.uint16,
    )
    occupied = np.zeros(sequence_length, dtype=bool)

    for interval in record["chord_intervals"]: 
        start = qstep(interval["onset_ql"], SELECTED_PPQ)
        end = qstep(interval["end_ql"], SELECTED_PPQ)
        if not 0 <= start < end <= sequence_length:
            raise ValueError(
                f"Invalid chord range: {start}:{end}/{sequence_length}"
            )
        if occupied[start:end].any(): 
            raise ValueError("24 PPQ chord state collision")

        
        
        mapped_label = map_chord_label(interval["label"]) 
        chord_id = chord_to_id[mapped_label]
        chord_grid[start:end] = chord_id
        occupied[start:end] = True

    if not occupied.all(): 
        raise ValueError('The song contains an uncovered chord timestep.')

    
    
    chord_onset_grid = np.zeros(sequence_length, dtype=np.uint8) 
    chord_onset_grid[0] = 1
    if sequence_length > 1:
        chord_onset_grid[1:] = (
            chord_grid[1:] != chord_grid[:-1]
        ).astype(np.uint8)

    unk_mask = chord_grid == CHORD_UNK_ID
    mapping_stats = {
        "original_chord_interval_count": int(
            len(record["chord_intervals"])
        ),
        "mapped_chord_onset_count": int(chord_onset_grid.sum()),
        "unk_grid_step_count": int(unk_mask.sum()),
        "unk_chord_onset_count": int(
            np.logical_and(unk_mask, chord_onset_grid == 1).sum()
        ),
    }
    return chord_grid, chord_onset_grid, mapping_stats

def encode_record_miditok(record): 
    sequence_length = qstep(
        record["score_length_ql"],
        SELECTED_PPQ,
    )
    if sequence_length <= 0:
        raise ValueError('The quantised song length is non-positive.')

    standard = standard_miditok_tsd(record)
    note_tokens_by_step = {} 
    for token_id, event_step, token_type in zip(
        standard["token_ids"],
        standard["event_steps"],
        standard["token_types"],
    ):
        if token_type not in {"Pitch", "Duration"}:
            continue
        step = int(event_step)
        if not 0 <= step < sequence_length:
            raise ValueError(
                f"MIDITok note-token time is outside the song range: {step}"
            )
        note_tokens_by_step.setdefault(step, []).append(int(token_id))

    chord_grid, chord_onset_grid, mapping_stats = (
        chord_grid_for_record(record, sequence_length)
    )

    input_ids = [MIDITOK_BOS_ID] 
    event_steps = [0]
    chord_target_ids = [CHORD_PAD_ID]
    chord_onset = [0]
    grid_mask = [0]

    for step in range(sequence_length): 
        for token_id in note_tokens_by_step.get(step, []):
            input_ids.append(token_id)
            event_steps.append(step)
            chord_target_ids.append(CHORD_PAD_ID)
            chord_onset.append(0)
            grid_mask.append(0)

        
        input_ids.append(MIDITOK_CLOCK_ID) 
        event_steps.append(step)
        chord_target_ids.append(int(chord_grid[step]))
        chord_onset.append(int(chord_onset_grid[step]))
        grid_mask.append(1)

    input_ids.append(MIDITOK_EOS_ID) 
    event_steps.append(sequence_length)
    chord_target_ids.append(CHORD_PAD_ID)
    chord_onset.append(0)
    grid_mask.append(0)

    encoded = {
        "input_ids": np.asarray(input_ids, dtype=np.int32),
        "event_steps": np.asarray(event_steps, dtype=np.int32),
        "chord_target_ids": np.asarray(
            chord_target_ids,
            dtype=np.uint16,
        ),
        "chord_onset": np.asarray(chord_onset, dtype=np.uint8),
        "grid_mask": np.asarray(grid_mask, dtype=np.uint8),
        "raw_input_ids": standard["token_ids"],
        "raw_event_steps": standard["event_steps"],
        "grid_steps": sequence_length,
        **mapping_stats,
    }

    aligned_lengths = {
        len(encoded["input_ids"]),
        len(encoded["event_steps"]),
        len(encoded["chord_target_ids"]),
        len(encoded["chord_onset"]),
        len(encoded["grid_mask"]),
    }
    if len(aligned_lengths) != 1:
        raise ValueError('MIDITok-aligned arrays have inconsistent lengths.')
    if int(encoded["grid_mask"].sum()) != sequence_length:
        raise ValueError('Not every 24-PPQ step has exactly one supervised slot.')
    if np.any(
        encoded["input_ids"][encoded["grid_mask"] == 1]
        != MIDITOK_CLOCK_ID
    ):
        raise ValueError('A supervised slot is not a MIDITok clock token.')

    supervised_targets = encoded["chord_target_ids"][
        encoded["grid_mask"] == 1
    ]
    if np.any(supervised_targets == CHORD_PAD_ID):
        raise ValueError('A supervised slot must not use the <PAD> chord target.')
    if supervised_targets.min() < CHORD_UNK_ID:
        raise ValueError('A supervised chord target ID is lower than the <UNK> ID.')
    if supervised_targets.max() >= len(chord_to_id):
        raise ValueError('A supervised chord target ID is outside the vocabulary.')
    if np.any(
        encoded["chord_target_ids"][encoded["grid_mask"] == 0]
        != CHORD_PAD_ID
    ):
        raise ValueError('A non-supervised token must not carry a gold chord label.')

    grid_onsets = encoded["chord_onset"][
        encoded["grid_mask"] == 1
    ]
    expected_onsets = np.zeros(sequence_length, dtype=np.uint8)
    expected_onsets[0] = 1
    if sequence_length > 1:
        expected_onsets[1:] = (
            supervised_targets[1:] != supervised_targets[:-1]
        ).astype(np.uint8)
    if not np.array_equal(grid_onsets, expected_onsets):
        raise ValueError('Chord onsets were not reconstructed correctly from the mapped target grid.')

    if np.any(encoded["input_ids"] == MIDITOK_PAD_ID):
        raise ValueError('Within-song MIDITok input must not contain PAD.')
    if len(note_tokens_by_step) == 0:
        raise ValueError('The MIDITok-aligned stream contains no melody-note tokens.')

    expected_aligned_length = (
        sequence_length
        + 2 * len(record["melody_events"])
        + 2
    )
    if len(encoded["input_ids"]) != expected_aligned_length:
        raise ValueError(
            'The aligned TSD token count is inconsistent; '
            f"expected={expected_aligned_length}, "
            f"actual={len(encoded['input_ids'])}"
        )
    return encoded

first_accepted_record = None
for batch_index in range(preprocess_batch_count):
    for record in load_json_gz(
        preprocess_paths(batch_index)["events"]
    ):
        if str(record["record_id"]) in accepted_ids:
            first_accepted_record = record
            break
    if first_accepted_record is not None:
        break

encoded_preflight = encode_record_miditok(first_accepted_record)
print("MIDITok encoding preflight:", first_accepted_record["record_id"])
print("Standard TSD tokens:", len(encoded_preflight["raw_input_ids"]))
print("Aligned TSD tokens:", len(encoded_preflight["input_ids"]))
print("24-PPQ supervised slots:", int(encoded_preflight["grid_mask"].sum()))
print("<UNK> supervised slots:", encoded_preflight["unk_grid_step_count"])
print(
    "Original / mapped chord onsets:",
    encoded_preflight["original_chord_interval_count"],
    "/",
    encoded_preflight["mapped_chord_onset_count"],
)
print("Gold chord tokens inside input: 0")

raw_preview = [
    miditok_tokenizer[int(token_id)]
    for token_id in encoded_preflight["raw_input_ids"][:24]
]
print("Standard MIDITok TSD preview:")
print(raw_preview)

preview_rows = []
for token_id, step, target_id, onset, is_grid in zip(
    encoded_preflight["input_ids"][:40],
    encoded_preflight["event_steps"][:40],
    encoded_preflight["chord_target_ids"][:40],
    encoded_preflight["chord_onset"][:40],
    encoded_preflight["grid_mask"][:40],
):
    preview_rows.append(
        {
            "miditok_token": miditok_tokenizer[int(token_id)],
            "step_24ppq": int(step),
            "quarterLength": round(int(step) / SELECTED_PPQ, 6),
            "supervised_slot": bool(is_grid),
            "chord_target": (
                id_to_chord[str(int(target_id))]
                if is_grid
                else ""
            ),
            "chord_onset": int(onset),
        }
    )
display(pd.DataFrame(preview_rows))
print("Preflight passed.")

## 21. Write resumable Top-N encoding shards

Each shard stores the standard MIDITok sequence, the aligned model stream, reduced chord targets, mapped chord onsets, grid masks and song offsets. The manifest records unknown-target counts and onset statistics.


In [ ]:
ENCODED_MANIFEST_COLUMNS = [
    "record_id", "source_index", "relative_path", "file_name",
    "group_id", "split", "shard_file", "shard_song_index",
    "offset_start", "offset_end", "aligned_token_count",
    "raw_offset_start", "raw_offset_end", "raw_token_count",
    "grid_steps", "melody_event_count", "chord_interval_count",
    "mapped_chord_onset_count", "unk_chord_onset_count",
    "unk_grid_step_count", "unk_grid_rate",
]

split_by_id = split_df.set_index("record_id")["split"].to_dict()

def encoding_paths(batch_index):
    stem = f"miditok_tsd24_topn_unk_{batch_index:04d}"
    return {
        "npz": ENCODING_SHARD_DIR / f"{stem}.npz",
        "manifest": ENCODING_SHARD_DIR / f"{stem}_manifest.csv",
        "done": ENCODING_SHARD_DIR / f"{stem}_done.json",
    }

def encoding_done_is_valid(paths, input_record_ids):
    if not all(
        paths[key].exists()
        for key in ["npz", "manifest", "done"]
    ):
        return False
    try:
        marker = json.loads(
            paths["done"].read_text(encoding="utf-8")
        )
        return (
            marker["preprocessing_revision"] == PIPELINE_REVISION
            and marker["encoding_revision"] == ENCODING_REVISION
            and marker["encoding_schema_sha256"]
            == ENCODING_SCHEMA_SHA256
            and marker["input_record_ids"] == input_record_ids
        )
    except Exception:
        return False

def save_npz_atomic(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)

for batch_index in tqdm(
    range(preprocess_batch_count),
    desc="MIDITok TSD-24 Top-N + <UNK> encoding shards",
):
    source_records = load_json_gz(
        preprocess_paths(batch_index)["events"]
    )
    accepted_records = [
        record
        for record in source_records
        if str(record["record_id"]) in accepted_ids
    ]
    input_record_ids = [
        str(record["record_id"]) for record in accepted_records
    ]
    paths = encoding_paths(batch_index)

    if encoding_done_is_valid(paths, input_record_ids):
        continue

    offsets = [0]
    raw_offsets = [0]
    manifest_rows = []
    array_lists = {
        "input_ids": [],
        "event_steps": [],
        "chord_target_ids": [],
        "chord_onset": [],
        "grid_mask": [],
        "raw_input_ids": [],
        "raw_event_steps": [],
    }

    for song_index, record in enumerate(accepted_records):
        encoded = encode_record_miditok(record)
        aligned_start = offsets[-1]
        aligned_end = aligned_start + len(encoded["input_ids"])
        raw_start = raw_offsets[-1]
        raw_end = raw_start + len(encoded["raw_input_ids"])
        offsets.append(aligned_end)
        raw_offsets.append(raw_end)

        for name in array_lists:
            array_lists[name].append(encoded[name])

        record_id = str(record["record_id"])
        grid_steps = int(encoded["grid_steps"])
        unk_grid_step_count = int(encoded["unk_grid_step_count"])
        manifest_rows.append(
            {
                "record_id": record_id,
                "source_index": int(record["source_index"]),
                "relative_path": record["relative_path"],
                "file_name": record["file_name"],
                "group_id": record["group_id"],
                "split": split_by_id[record_id],
                "shard_file": paths["npz"].name,
                "shard_song_index": song_index,
                "offset_start": aligned_start,
                "offset_end": aligned_end,
                "aligned_token_count": len(encoded["input_ids"]),
                "raw_offset_start": raw_start,
                "raw_offset_end": raw_end,
                "raw_token_count": len(encoded["raw_input_ids"]),
                "grid_steps": grid_steps,
                "melody_event_count": len(record["melody_events"]),
                "chord_interval_count": len(
                    record["chord_intervals"]
                ),
                "mapped_chord_onset_count": int(
                    encoded["mapped_chord_onset_count"]
                ),
                "unk_chord_onset_count": int(
                    encoded["unk_chord_onset_count"]
                ),
                "unk_grid_step_count": unk_grid_step_count,
                "unk_grid_rate": (
                    unk_grid_step_count / grid_steps
                    if grid_steps else 0.0
                ),
            }
        )

    def concatenate_or_empty(name, dtype):
        arrays = array_lists[name]
        return (
            np.concatenate(arrays).astype(dtype, copy=False)
            if arrays
            else np.asarray([], dtype=dtype)
        )

    save_npz_atomic(
        paths["npz"],
        input_ids=concatenate_or_empty("input_ids", np.int32),
        event_steps=concatenate_or_empty("event_steps", np.int32),
        chord_target_ids=concatenate_or_empty(
            "chord_target_ids", np.uint16
        ),
        chord_onset=concatenate_or_empty(
            "chord_onset", np.uint8
        ),
        grid_mask=concatenate_or_empty("grid_mask", np.uint8),
        raw_input_ids=concatenate_or_empty(
            "raw_input_ids", np.int32
        ),
        raw_event_steps=concatenate_or_empty(
            "raw_event_steps", np.int32
        ),
        offsets=np.asarray(offsets, dtype=np.int64),
        raw_offsets=np.asarray(raw_offsets, dtype=np.int64),
        record_ids=np.asarray(input_record_ids, dtype="<U64"),
        splits=np.asarray(
            [split_by_id[record_id] for record_id in input_record_ids],
            dtype="<U10",
        ),
    )
    shard_manifest_df = pd.DataFrame(
        manifest_rows,
        columns=ENCODED_MANIFEST_COLUMNS,
    )
    atomic_csv(shard_manifest_df, paths["manifest"])
    atomic_json(
        {
            "preprocessing_revision": PIPELINE_REVISION,
            "encoding_revision": ENCODING_REVISION,
            "encoding_schema_sha256": ENCODING_SCHEMA_SHA256,
            "batch_index": batch_index,
            "input_record_ids": input_record_ids,
            "song_count": len(input_record_ids),
            "aligned_token_count": int(offsets[-1]),
            "raw_token_count": int(raw_offsets[-1]),
            "grid_step_count": int(
                sum(row["grid_steps"] for row in manifest_rows)
            ),
            "unk_grid_step_count": int(
                sum(row["unk_grid_step_count"] for row in manifest_rows)
            ),
            "mapped_chord_onset_count": int(
                sum(
                    row["mapped_chord_onset_count"]
                    for row in manifest_rows
                )
            ),
        },
        paths["done"],
    )

print("All Top-N + <UNK> MIDITok encoding shards reached done marker.")

## 22. Final encoding audit

The audit checks sequence boundaries, vocabulary ranges, clock counts, masks, target placement, mapped onsets, unknown totals, group separation and the absence of gold chord information from model input.


In [ ]:
encoded_manifest_frames = []
total_aligned_tokens = 0
total_raw_tokens = 0
total_grid_slots = 0
total_unk_slots = 0
total_mapped_chord_onsets = 0
total_unk_chord_onsets = 0
shard_song_count = 0

for batch_index in tqdm(
    range(preprocess_batch_count),
    desc="Final Top-N + <UNK> encoding validation",
):
    paths = encoding_paths(batch_index)
    if not all(
        paths[key].exists()
        for key in ["npz", "manifest", "done"]
    ):
        raise RuntimeError(
            f"MIDITok encoding batch {batch_index} has not been saved completely"
        )

    manifest_part = pd.read_csv(paths["manifest"])
    encoded_manifest_frames.append(manifest_part)

    with np.load(paths["npz"], allow_pickle=False) as shard:
        aligned_lengths = {
            len(shard["input_ids"]),
            len(shard["event_steps"]),
            len(shard["chord_target_ids"]),
            len(shard["chord_onset"]),
            len(shard["grid_mask"]),
        }
        assert len(aligned_lengths) == 1
        aligned_length = aligned_lengths.pop()
        raw_lengths = {
            len(shard["raw_input_ids"]),
            len(shard["raw_event_steps"]),
        }
        assert len(raw_lengths) == 1
        raw_length = raw_lengths.pop()

        offsets = shard["offsets"]
        raw_offsets = shard["raw_offsets"]
        record_ids = shard["record_ids"].astype(str)
        assert offsets[0] == 0
        assert raw_offsets[0] == 0
        assert offsets[-1] == aligned_length
        assert raw_offsets[-1] == raw_length
        assert len(offsets) == len(record_ids) + 1
        assert len(raw_offsets) == len(record_ids) + 1
        assert len(record_ids) == len(manifest_part)
        assert record_ids.tolist() == (
            manifest_part["record_id"].astype(str).tolist()
        )

        for song_index, row in manifest_part.reset_index(
            drop=True
        ).iterrows():
            start = int(offsets[song_index])
            end = int(offsets[song_index + 1])
            raw_start = int(raw_offsets[song_index])
            raw_end = int(raw_offsets[song_index + 1])
            assert end > start
            assert raw_end > raw_start
            assert end - start == int(row["aligned_token_count"])
            assert raw_end - raw_start == int(row["raw_token_count"])

            input_ids = shard["input_ids"][start:end]
            event_steps = shard["event_steps"][start:end]
            targets = shard["chord_target_ids"][start:end]
            onset = shard["chord_onset"][start:end]
            mask = shard["grid_mask"][start:end]
            raw_ids = shard["raw_input_ids"][raw_start:raw_end]
            raw_steps = shard["raw_event_steps"][raw_start:raw_end]

            assert int(input_ids[0]) == MIDITOK_BOS_ID
            assert int(input_ids[-1]) == MIDITOK_EOS_ID
            assert MIDITOK_PAD_ID not in input_ids
            assert MIDITOK_PAD_ID not in raw_ids
            assert input_ids.min() >= 0
            assert input_ids.max() < len(miditok_tokenizer)
            assert raw_ids.min() >= 0
            assert raw_ids.max() < len(miditok_tokenizer)
            assert set(np.unique(mask)).issubset({0, 1})
            assert set(np.unique(onset)).issubset({0, 1})
            assert int(mask.sum()) == int(row["grid_steps"])
            assert np.all(input_ids[mask == 1] == MIDITOK_CLOCK_ID)

            supervised_targets = targets[mask == 1]
            supervised_onsets = onset[mask == 1]
            assert np.all(supervised_targets != CHORD_PAD_ID)
            assert supervised_targets.min() >= CHORD_UNK_ID
            assert supervised_targets.max() < len(chord_to_id)
            assert np.all(targets[mask == 0] == CHORD_PAD_ID)
            assert np.all(onset[mask == 0] == 0)
            assert np.all(np.diff(event_steps) >= 0)
            assert np.all(np.diff(raw_steps) >= 0)

            expected_onsets = np.zeros(
                len(supervised_targets), dtype=np.uint8
            )
            expected_onsets[0] = 1
            if len(supervised_targets) > 1:
                expected_onsets[1:] = (
                    supervised_targets[1:]
                    != supervised_targets[:-1]
                ).astype(np.uint8)
            assert np.array_equal(supervised_onsets, expected_onsets)

            song_unk_slots = int(
                (supervised_targets == CHORD_UNK_ID).sum()
            )
            song_mapped_onsets = int(supervised_onsets.sum())
            song_unk_onsets = int(
                np.logical_and(
                    supervised_targets == CHORD_UNK_ID,
                    supervised_onsets == 1,
                ).sum()
            )
            assert song_unk_slots == int(row["unk_grid_step_count"])
            assert song_mapped_onsets == int(
                row["mapped_chord_onset_count"]
            )
            assert song_unk_onsets == int(
                row["unk_chord_onset_count"]
            )

            total_unk_slots += song_unk_slots
            total_mapped_chord_onsets += song_mapped_onsets
            total_unk_chord_onsets += song_unk_onsets

        total_aligned_tokens += int(aligned_length)
        total_raw_tokens += int(raw_length)
        total_grid_slots += int(shard["grid_mask"].sum())
        shard_song_count += len(record_ids)

encoding_manifest_df = pd.concat(
    encoded_manifest_frames,
    ignore_index=True,
).sort_values("source_index").reset_index(drop=True)

assert encoding_manifest_df["record_id"].is_unique
assert set(encoding_manifest_df["record_id"].astype(str)) == accepted_ids
assert len(encoding_manifest_df) == len(accepted_df)
assert shard_song_count == len(accepted_df)
assert int(encoding_manifest_df["aligned_token_count"].sum()) == (
    total_aligned_tokens
)
assert int(encoding_manifest_df["raw_token_count"].sum()) == (
    total_raw_tokens
)
assert int(encoding_manifest_df["grid_steps"].sum()) == (
    total_grid_slots
)
assert int(encoding_manifest_df["unk_grid_step_count"].sum()) == (
    total_unk_slots
)
assert int(
    encoding_manifest_df["mapped_chord_onset_count"].sum()
) == total_mapped_chord_onsets
assert int(
    encoding_manifest_df["unk_chord_onset_count"].sum()
) == total_unk_chord_onsets
assert (
    encoding_manifest_df.groupby("group_id")["split"]
    .nunique()
    .max()
    == 1
)


for coverage_row in coverage_by_split_records:
    split_name = coverage_row["split"]
    manifest_split = encoding_manifest_df[
        encoding_manifest_df["split"].eq(split_name)
    ]
    assert int(manifest_split["grid_steps"].sum()) == int(
        coverage_row["total_slots"]
    )
    assert int(manifest_split["unk_grid_step_count"].sum()) == int(
        coverage_row["unk_slots"]
    )

ENCODING_MANIFEST_PATH = (
    ENCODING_ROOT
    / "xml_only_miditok_tsd24_topn_unk_v3_manifest.csv"
)
ENCODING_SUMMARY_PATH = (
    ENCODING_ROOT
    / "xml_only_miditok_tsd24_topn_unk_v3_summary.json"
)
atomic_csv(encoding_manifest_df, ENCODING_MANIFEST_PATH)

split_counts = {
    key: int(value)
    for key, value in encoding_manifest_df[
        "split"
    ].value_counts().to_dict().items()
}
original_chord_onset_count = int(
    encoding_manifest_df["chord_interval_count"].sum()
)
onset_removed_by_mapping_count = (
    original_chord_onset_count - total_mapped_chord_onsets
)

encoding_summary = {
    "preprocessing_revision": PIPELINE_REVISION,
    "encoding_revision": ENCODING_REVISION,
    "data_source": "single_mxl_only",
    "jams_used": False,
    "ppq": SELECTED_PPQ,
    "miditok_used": True,
    "miditok_version": importlib.metadata.version("miditok"),
    "symusic_version": importlib.metadata.version("symusic"),
    "tokenizer": "TSD",
    "representation": "MIDITok TSD-24 alignment-preserving expansion",
    "miditok_use_chords": False,
    "chord_target_source": "same MXL harmony converted to Harte",
    "chord_reduction_strategy": VOCAB_REDUCTION_STRATEGY,
    "vocabulary_selected_from_split": "train",
    "vocabulary_ranking_basis": VOCAB_RANKING_BASIS,
    "requested_top_n": int(TOP_N_RETAINED_CHORDS),
    "raw_training_chord_label_count": int(len(training_slot_counts)),
    "retained_chord_label_count": int(len(retained_chord_labels)),
    "supervised_chord_class_count_including_unk": int(
        len(retained_chord_labels) + 1
    ),
    "chord_vocab_size_including_pad": int(len(chord_to_id)),
    
    "chord_vocab_size": int(len(chord_to_id)),
    "coverage_by_split": coverage_by_split_records,
    "gold_chord_token_inside_input_count": 0,
    "song_count": int(len(encoding_manifest_df)),
    "standard_miditok_token_count": int(total_raw_tokens),
    "aligned_miditok_token_count": int(total_aligned_tokens),
    "supervised_grid_slot_count": int(total_grid_slots),
    "one_supervised_slot_per_grid_step": True,
    "miditok_vocab_size": int(len(miditok_tokenizer)),
    "split_song_counts": split_counts,
    "song_group_leakage_count": 0,
    "input_pad_inside_song_count": 0,
    "unknown_chord_target_allowed": True,
    "unknown_chord_on_supervised_slot_count": int(total_unk_slots),
    "unknown_chord_on_supervised_slot_rate": (
        float(total_unk_slots / total_grid_slots)
        if total_grid_slots else 0.0
    ),
    "original_chord_onset_count": original_chord_onset_count,
    "mapped_chord_onset_count": int(total_mapped_chord_onsets),
    "unknown_chord_onset_count": int(total_unk_chord_onsets),
    "onset_removed_by_mapping_count": int(
        onset_removed_by_mapping_count
    ),
    "onset_recomputed_after_mapping": True,
    "array_length_mismatch_count": 0,
    "pending_count": 0,
    "model_input_array": "input_ids",
    "model_target_array": "chord_target_ids",
    "model_loss_mask_array": "grid_mask",
    "ready_for_lstm_windows": True,
    "preprocessing_summary_path": str(PREPROCESS_SUMMARY_PATH),
    "miditok_tokenizer_path": str(MIDITOK_TOKENIZER_PATH),
    "vocab_path": str(VOCAB_PATH),
    "split_manifest_path": str(SPLIT_PATH),
    "chord_coverage_path": str(CHORD_COVERAGE_PATH),
    "chord_frequency_path": str(TRAIN_CHORD_FREQUENCY_PATH),
    "chord_mapping_path": str(CHORD_MAPPING_PATH),
    "split_coverage_path": str(SPLIT_COVERAGE_PATH),
    "encoding_manifest_path": str(ENCODING_MANIFEST_PATH),
    "shard_directory": str(ENCODING_SHARD_DIR),
}
atomic_json(encoding_summary, ENCODING_SUMMARY_PATH)

print(json.dumps(encoding_summary, indent=2))
print("\nTOP-N + <UNK> MIDITOK ENCODING COMPLETE ✅")

## 23. Produce the evidence table used for reporting


In [ ]:
train_coverage = next(
    row
    for row in coverage_by_split_records
    if row["split"] == "train"
)
test_coverage = next(
    row
    for row in coverage_by_split_records
    if row["split"] == "test"
)

meeting_evidence = pd.DataFrame(
    [
        {
            "evidence": "Source",
            "result": "One MXL per song; melody + annotated harmony from same part",
        },
        {
            "evidence": "JAMS used",
            "result": False,
        },
        {
            "evidence": "Vocabulary selection data",
            "result": "Training split only",
        },
        {
            "evidence": "Ranking basis",
            "result": VOCAB_RANKING_BASIS,
        },
        {
            "evidence": "Raw training chord labels",
            "result": encoding_summary[
                "raw_training_chord_label_count"
            ],
        },
        {
            "evidence": "Retained chord labels",
            "result": encoding_summary[
                "retained_chord_label_count"
            ],
        },
        {
            "evidence": "Supervised classes",
            "result": (
                f"{encoding_summary['retained_chord_label_count']} retained "
                "+ 1 <UNK>"
            ),
        },
        {
            "evidence": "Training slot coverage",
            "result": f"{train_coverage['retained_slot_coverage_pct']:.4f}%",
        },
        {
            "evidence": "Training chord-event coverage",
            "result": (
                f"{train_coverage['retained_chord_event_coverage_pct']:.4f}%"
            ),
        },
        {
            "evidence": "Test <UNK> slot rate",
            "result": f"{test_coverage['unk_slot_pct']:.4f}%",
        },
        {
            "evidence": "Onsets before / after mapping",
            "result": (
                f"{encoding_summary['original_chord_onset_count']} / "
                f"{encoding_summary['mapped_chord_onset_count']}"
            ),
        },
        {
            "evidence": "Onset rebuilt after mapping",
            "result": encoding_summary[
                "onset_recomputed_after_mapping"
            ],
        },
        {
            "evidence": "MIDITok used",
            "result": encoding_summary["miditok_used"],
        },
        {
            "evidence": "Formal tokenizer",
            "result": (
                f"MIDITok {encoding_summary['miditok_version']} "
                "TSD (TimeShift / Pitch / Duration)"
            ),
        },
        {
            "evidence": "MIDITok chord detection",
            "result": "Disabled; preserves MXL annotated Harte targets",
        },
        {
            "evidence": "Linear alignment",
            "result": "One TimeShift_0.1.24 supervised clock per 24-PPQ step",
        },
        {
            "evidence": "Gold chord leakage into input",
            "result": 0,
        },
        {
            "evidence": "MXL attempted",
            "result": preprocessing_summary["source_mxl_count"],
        },
        {
            "evidence": "Accepted / encoded songs",
            "result": encoding_summary["song_count"],
        },
        {
            "evidence": "Rejected songs",
            "result": preprocessing_summary["rejected_count"],
        },
        {
            "evidence": "12 / 24 PPQ unsafe songs",
            "result": (
                f"{preprocessing_summary['q12_unsafe_song_count']} / "
                f"{preprocessing_summary['q24_unsafe_song_count']}"
            ),
        },
        {
            "evidence": "Song-group leakage",
            "result": 0,
        },
        {
            "evidence": "Ready for renewed LSTM windows",
            "result": encoding_summary[
                "ready_for_lstm_windows"
            ],
        },
    ]
)
display(meeting_evidence)

print("Key files:")
for path in [
    PREPROCESS_MANIFEST_PATH,
    PREPROCESS_SUMMARY_PATH,
    CHORD_COVERAGE_PATH,
    TRAIN_CHORD_FREQUENCY_PATH,
    CHORD_MAPPING_PATH,
    SPLIT_COVERAGE_PATH,
    MIDITOK_TOKENIZER_PATH,
    VOCAB_PATH,
    SPLIT_PATH,
    ENCODING_MANIFEST_PATH,
    ENCODING_SUMMARY_PATH,
]:
    print("-", path)

## Encoding-stage boundary

This notebook ends with audited input IDs, reduced chord targets, grid masks and chord-onset indicators. Model-window construction, training and evaluation are implemented in the subsequent notebooks.
